In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
import itertools
import random
import pandas as pd
import numpy as np
from uuid import uuid4

Exception: The official Pinecone python package has been renamed from `pinecone-client` to `pinecone`. Please remove `pinecone-client` from your project dependencies and add `pinecone` instead. See the README at https://github.com/pinecone-io/pinecone-python-client for more information on using the python SDK.

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

client = OpenAI(api_key=openai_api_key)
pc = Pinecone(api_key=pinecone_api_key)

In [ ]:
index_name = "pc-3"

In [ ]:
index = pc.Index(index_name, pool_threads = 10)

In [ ]:
youtube_df = pd.read_csv("youtube_rag_data_small.csv")
type(youtube_df)

batch_limit = 100
count = 0

for i in range(0, len(youtube_df), batch_limit):
    batch = youtube_df.iloc[i:i+batch_limit]

    metadatas = [{
        "text_id": row['id'],
        "text": row['text'],
        "title": row['title'],
        "url": row['url'],
        "published": row['published']
    } for _, row in batch.iterrows()]

    texts = batch['text'].tolist()
    ids = [str(uuid4()) for _ in range(len(texts))]

    response = client.embeddings.create(
        input=texts,
        model="text-embedding-3-small"
    )

    embeds = [list(x.embedding) for x in response.data]

    vectors = [
        {
            "id": ids[i],
            "values": embeds[i],
            "metadata": metadatas[i]
        }
        for i in range(len(ids))
    ]

    index.upsert(vectors=vectors, namespace='youtube_rag_dataset')

    count += 1
    print(f"Batch {count} uploaded")

print(index.describe_index_stats())


In [ ]:
def retrieve(
    query,
    top_k=5,
    namespace="youtube_rag_dataset",
    emb_model="text-embedding-3-small",
    score_threshold=0.0
):
    response = client.embeddings.create(
        input=query,
        model=emb_model
    )

    query_emb = list(response.data[0].embedding)

    results = index.query(
        vector=query_emb,
        top_k=top_k,
        namespace=namespace,
        include_metadata=True
    )

    matches = results.get("matches", [])

    docs = []
    sources = []

    for match in matches:
        score = match.get("score", 0)

        if score < score_threshold:
            continue

        metadata = match.get("metadata", {})

        text = metadata.get("text", "")
        title = metadata.get("title", "Unknown Title")
        url = metadata.get("url", "No URL")

        if text:
            docs.append(text)
            sources.append((title, url))

    if not docs:
        print("No relevant documents found.")

    return docs, sources

In [ ]:
docs, sources = retrieve("What is a transformer model?", top_k=3)
print(docs)
print(sources)

In [ ]:
def prompt_with_context_builder(query, docs, sources=None):
    delim = "\n\n---\n\n"
    context = delim.join(docs)

    prompt = f"""
You are a precise AI assistant.

Use ONLY the context below.
If answer is not present, say:
"I don't know based on the provided context."

Context:
{context}

Question: {query}

Answer:
"""
    return prompt.strip()

In [ ]:
def question_answering(prompt, sources, chat_model):
    system_prompt = """
You are a precise and reliable AI assistant.

Rules:
- Answer ONLY using the provided context
- If the answer is not in the context, say:
  "I don't know based on the provided context."
- Do NOT make up information
- Keep answers concise and clear
"""

    if not prompt or "Context:\n\n" in prompt:
        return "No relevant context found to answer the question."

    res = client.chat.completions.create(
        model=chat_model,
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    answer = res.choices[0].message.content.strip()

    if sources:
        source_text = "\n\nSources:\n"
        for i, (title, url) in enumerate(sources):
            source_text += f"{i+1}. {title}\n   {url}\n"
        answer += source_text

    return answer

In [ ]:
query = "How to build next-level Q&A with OpenAI"

documents, sources = retrieve(
    query,
    top_k=3,
    namespace='youtube_rag_dataset',
    emb_model="text-embedding-3-small"
)

prompt = prompt_with_context_builder(query, documents, sources)

answer = question_answering(
    prompt=prompt,
    sources=sources,
    chat_model='gpt-4o-mini'
)

print(answer)